# memory MCP 서버 + LangChain 에이전트

`@modelcontextprotocol/server-memory`(stdio)를 LangChain 에이전트의 도구로 연결하는 예제.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so you can open that file and read the graph it builds.

That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.

## 1. 세팅

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


##  메모리 파일 경로와 프롬프트 준비

In [3]:
import os

# 메모리 파일 경로를 절대경로로 만든다
memory_path = os.path.abspath("memory/memory.json")
# 상위 폴더가 없으면 서버가 쓰기에 실패하므로 미리 만들어 둔다 (있으면 통과)
os.makedirs(os.path.dirname(memory_path), exist_ok=True)


##  MCP 서버 연결 및 도구 로드

In [4]:
# 여러 MCP 서버에 연결하고 도구를 가져오는 클라이언트
from langchain_mcp_adapters.client import MultiServerMCPClient

# memory MCP 서버를 stdio(표준 입출력) 방식의 서브프로세스로 실행하도록 설정한다
client = MultiServerMCPClient(
    {
        "memory": {
            "transport": "stdio",              # 로컬 프로세스와 표준 입출력으로 통신
            "command": "npx",                  # 서버를 실행할 명령
            "args": ["-y", "@modelcontextprotocol/server-memory"],
            # 부모 환경변수를 그대로 물려주고 MEMORY_FILE_PATH만 덮어쓴다 (PATH 유실 방지)
            "env": {**os.environ, "MEMORY_FILE_PATH": memory_path},
        }
    }
)

# 서버의 MCP 도구들을 LangChain 도구 객체로 변환해서 가져온다
tools = await client.get_tools()

# 로드된 도구 이름 목록을 확인한다
[t.name for t in tools]

['create_entities',
 'create_relations',
 'add_observations',
 'delete_entities',
 'delete_observations',
 'delete_relations',
 'read_graph',
 'search_nodes',
 'open_nodes']

##  에이전트 생성 및 실행

In [5]:
# 에이전트에게 부여할 시스템 지침
instructions = "너는 엔티티 도구를 영속적인 메모리로 사용해서 대화에 관한 정보를 저장하고 다시 불러온다."

# 사용자 요청문
request = "제 이름은 Ed입니다. 저는 LLM 엔지니어예요. AI 에이전트 강의를 하고 있습니다."

In [6]:
# 도구를 사용하는 에이전트를 생성하는 함수
from langchain.agents import create_agent

# 모델 + 도구 + 시스템 지침으로 에이전트를 만든다
agent = create_agent("openai:gpt-5-mini", tools, system_prompt=instructions)

# messages 키에 사용자 입력을 넣어 비동기로 실행한다
result = await agent.ainvoke({"messages": request})

# 마지막 메시지가 에이전트의 최종 답변이다
print(result["messages"][-1].content)

반가워요, Ed — 정보(이름: Ed, 직업: LLM 엔지니어, 강의: AI 에이전트 강의)를 메모리에 저장해뒀습니다.

어떻게 도와드릴까요? 예시 제안들:
- 강의 커리큘럼/주차별 강의안 초안 작성
- 슬라이드 개요 및 핵심 슬라이드 문구 작성
- 실습(코드/노트북) 예제와 과제 제작
- 평가문제(퀴즈/과제/프로젝트) 설계
- 학생 대상 데모 에이전트(예: ReAct, Reflexion, Tool-using) 코드 샘플
- 강의용 프롬프트 템플릿 및 실습 가이드

추가로 메모리에 저장할 정보가 있나요? (예: 대상 학생 수준, 강의 기간/주수, 사용 언어/툴스택, 소속 기관, 선호하는 교수법 등)


In [7]:
result2 = await agent.ainvoke({"messages": "제 이름은 Ed입니다. 저에 대해 무엇을 알고 있나요?"})

# 마지막 메시지가 에이전트의 최종 답변이다
print(result2["messages"][-1].content)

안녕하세요 Ed님. 지금 제 기억(저장된 정보)에 있는 내용은 다음과 같습니다.

- 이름: Ed  
- 직업: LLM 엔지니어  
- 가르치는 것: AI 에이전트 관련 강의(강좌)

현재 이 엔티티와 연결된 별도 관계는 저장되어 있지 않습니다.

이 정보에 수정하거나 추가하고 싶은 내용(예: 연락처, 관심사, 현재 프로젝트 등)이 있으면 알려주세요. 원하시면 바로 메모에 추가·수정해 드리겠습니다.


##  저장 결과 확인

In [8]:
# 메모리 파일이 실제로 생성되었는지 확인한다
print(os.path.exists(memory_path))

# 파일이 있으면 내용을 그대로 출력한다 (한 줄에 JSON 하나씩 들어가는 JSONL 형식)
if os.path.exists(memory_path):
    print(open(memory_path, encoding="utf-8").read())

True
{"type":"entity","name":"Ed","entityType":"Person","observations":["Name: Ed","Profession: LLM engineer","Teaches: AI agent lectures"]}
{"type":"entity","name":"LLM engineer","entityType":"Profession","observations":["Works with large language models (LLMs)","Role: builds and tunes LLMs, designs prompts and systems"]}
{"type":"entity","name":"AI agent lectures","entityType":"Course","observations":["Topic: AI agents","Format: lectures/courses taught by Ed","Audience: students and practitioners interested in building agents"]}
